# AEFS Extended Evaluation: BorutaSHAP + SVM

Adds two new comparisons to the original 5-method evaluation:
- **BorutaSHAP** (Kartika et al., 2026): SHAP-based shadow feature selection
- **SVM** classifier: non-tree-based model to broaden threat model

Runs on all 4 pools: AEEEM, TeraPromise (15-project subsample), NASA MDP, ReLink.

**Setup:**
1. Upload `AEFS_data.zip` as dataset (must include `aeeem/`, `tera/`, `nasa/`, `relink/`)
2. Add `HF_TOKEN` to Secrets
3. Run all cells

In [ ]:
!pip install -q lightgbm shap imbalanced-learn scikit-learn tqdm huggingface_hub

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, time, math, re
import numpy as np
import pandas as pd
from pathlib import Path
from itertools import combinations
from scipy import stats
from scipy.stats import pointbiserialr
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif, RFE
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import matthews_corrcoef, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import shap
import urllib.request
from sklearn.model_selection import StratifiedKFold
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

def _cv_mcc(X, y, k=3, rs=42):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=rs)
    scores = []
    for tri, vali in skf.split(X, y):
        c = LGBMClassifier(**LGBM_SFS)
        c.fit(X.iloc[tri], y.iloc[tri])
        scores.append(matthews_corrcoef(y.iloc[vali], c.predict(X.iloc[vali])))
    return float(np.mean(scores))

def sequential_forward_selection(X, y, candidates, cv=3, min_imp=0.001, max_feat=None, min_feat=5):
    remaining = list(candidates)
    selected, best_score, history = [], -1.0, []
    max_feat = max_feat or len(remaining)
    while remaining and len(selected) < max_feat:
        scores = {f: _cv_mcc(X[selected+[f]], y, k=cv) for f in remaining}
        bf = max(scores, key=scores.get)
        bs = scores[bf]
        if bs - best_score < min_imp and selected:
            break
        selected.append(bf); remaining.remove(bf); best_score = bs
        history.append((bf, bs))
    if len(selected) < min_feat and remaining:
        for f in list(remaining):
            if len(selected) >= min_feat: break
            selected.append(f); remaining.remove(f)
    return selected

print('Setup complete.')

In [ ]:
RAW_AEEEM_DIR = Path('/kaggle/input/datasets/afrojahaqueshornaly/aefs-data/aeeem')
RAW_TERA_DIR = Path('/kaggle/input/datasets/afrojahaqueshornaly/aefs-data/tera')
RAW_NASA_DIR = Path('/kaggle/input/datasets/afrojahaqueshornaly/aefs-data/nasa')
RAW_RELINK_DIR = Path('/kaggle/input/datasets/afrojahaqueshornaly/aefs-data/relink')
GITHUB_BASE = 'https://raw.githubusercontent.com/bharlow058/AEEEM-and-other-SDP-datasets/master/datasetcsv'

def auto_download(raw_dir, github_subdir, files):
    if raw_dir.exists() and len(list(raw_dir.glob('*.csv'))) > 0:
        return
    print(f'Downloading from GitHub: {github_subdir}')
    raw_dir.mkdir(parents=True, exist_ok=True)
    for f in files:
        urllib.request.urlretrieve(f'{GITHUB_BASE}/{github_subdir}/{f}', raw_dir / f)

Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)
RESULTS_DIR = '/kaggle/working/results'

## 1. Data Loaders

In [ ]:
def _sanitize_cols(df):
    df.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', c).strip('_') for c in df.columns]
    seen = {}
    new_cols = []
    for c in df.columns:
        if c in seen:
            seen[c] += 1
            new_cols.append(f'{c}_{seen[c]}')
        else:
            seen[c] = 0
            new_cols.append(c)
    df.columns = new_cols
    return df

def load_aeeem():
    auto_download(RAW_AEEEM_DIR, 'AEEEM', ['Equinox.csv','JDT.csv','Lucene.csv','Mylyn.csv','PDE.csv'])
    pool = {}
    for f in sorted(RAW_AEEEM_DIR.glob('*.csv')):
        df = pd.read_csv(f)
        if 'class' in df.columns:
            df['defective'] = df['class'].map({'buggy': 1, 'clean': 0})
            df = df.drop(columns=['class'])
        df = df.select_dtypes(include=[np.number])
        df = _sanitize_cols(df).dropna().drop_duplicates()
        if len(df) >= 20: pool[f.stem.lower()] = df
    return pool

def load_tera():
    auto_download(RAW_TERA_DIR, 'TeraPromise',
        ['ant-1.3.csv','ant-1.4.csv','ant-1.5.csv','ant-1.6.csv','ant-1.7.csv',
         'camel-1.0.csv','camel-1.2.csv','camel-1.4.csv','camel-1.6.csv',
         'ivy-1.1.csv','ivy-1.4.csv','ivy-2.0.csv',
         'jedit-3.1.csv','jedit-3.2.csv','jedit-4.0.csv','jedit-4.1.csv','jedit-4.2.csv','jedit-4.3.csv',
         'log4j-1.0.csv','log4j-1.1.csv','log4j-1.2.csv',
         'lucene-2.0.csv','lucene-2.2.csv','lucene-2.4.csv',
         'poi-1.5.csv','poi-2.0.csv','poi-2.5.csv','poi-3.0.csv',
         'redaktor-1.0.csv',
         'synapse-1.0.csv','synapse-1.1.csv','synapse-1.2.csv',
         'xalan-2.4.csv','xalan-2.5.csv','xalan-2.6.csv','xalan-2.7.csv',
         'xerces-1.1.csv','xerces-1.2.csv','xerces-1.3.csv','xerces-1.4.csv'])
    pool = {}
    for f in sorted(RAW_TERA_DIR.glob('*.csv')):
        df = pd.read_csv(f)
        if 'bug' in df.columns:
            df['defective'] = (df['bug'] > 0).astype(int)
            df = df.drop(columns=['bug'])
        for dc in ['name','name.1','version']:
            if dc in df.columns: df = df.drop(columns=[dc])
        df = df.select_dtypes(include=[np.number])
        df = _sanitize_cols(df).dropna().drop_duplicates()
        if len(df) >= 20 and 'defective' in df.columns: pool[f.stem.lower()] = df
    return pool

def load_nasa():
    auto_download(RAW_NASA_DIR, 'NASA', ['PC3.csv','PC4.csv','cm1.csv','mw1.csv'])
    tmap = {'Y':1,'N':0,'TRUE':1,'FALSE':0,'true':1,'false':0}
    pool = {}
    for f in sorted(RAW_NASA_DIR.glob('*.csv')):
        name = f.stem.lower()
        if name in ('pc1','pc5','pc6'): continue
        df = pd.read_csv(f)
        tc = next((c for c in ['Defective','c','defects'] if c in df.columns), None)
        if tc is None: continue
        df['defective'] = df[tc].astype(str).str.strip().str.upper().map(tmap)
        df = df.drop(columns=[tc]).select_dtypes(include=[np.number])
        df = _sanitize_cols(df).dropna().drop_duplicates()
        if len(df) >= 20 and df['defective'].nunique() == 2: pool[name] = df
    return pool

def load_relink():
    auto_download(RAW_RELINK_DIR, 'Relink', ['apache.csv','safe.csv','zxing.csv'])
    pool = {}
    for f in sorted(RAW_RELINK_DIR.glob('*.csv')):
        df = pd.read_csv(f)
        if 'isDefective' in df.columns:
            df['defective'] = df['isDefective'].map({'buggy':1,'clean':0})
            df = df.drop(columns=['isDefective'])
        df = df.select_dtypes(include=[np.number])
        df = _sanitize_cols(df).dropna().drop_duplicates()
        if len(df) >= 20: pool[f.stem.lower()] = df
    return pool

aeeem = load_aeeem()
tera_full = load_tera()
nasa = load_nasa()
relink = load_relink()

# Subsample TeraPromise to 15 representative projects (full pool too slow for Kaggle 12h limit)
TERA_SUBSET = ['ant-1.7','camel-1.6','ivy-2.0','jedit-4.3','log4j-1.2',
               'lucene-2.4','poi-3.0','synapse-1.2','xalan-2.7','xerces-1.4',
               'ant-1.3','camel-1.0','jedit-3.1','log4j-1.0','lucene-2.0']
tera = {k: v for k, v in tera_full.items() if k in TERA_SUBSET}
print(f'AEEEM: {len(aeeem)}  Tera: {len(tera)} (subsampled from {len(tera_full)})  NASA: {len(nasa)}  ReLink: {len(relink)}')

## 2. Pipeline Functions

In [ ]:
LGBM_SFS = dict(n_estimators=60, max_depth=5, num_leaves=15, verbosity=-1, n_jobs=-1, random_state=RANDOM_STATE)
LGBM_FINAL = dict(n_estimators=300, max_depth=-1, num_leaves=31, learning_rate=0.05, verbosity=-1, n_jobs=-1, random_state=RANDOM_STATE)
RF_FINAL = dict(n_estimators=300, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE)
SVM_FINAL = dict(kernel='rbf', probability=True, random_state=RANDOM_STATE)

def rup_filter(df, vt=1e-4, rt=0.02, ct=0.90):
    cols = [c for c in df.columns if c != 'defective']
    X, y = df[cols], df['defective']
    kept = [c for c, s in zip(cols, VarianceThreshold(threshold=vt).fit(X).get_support()) if s]
    rel = [c for c in kept if abs(pointbiserialr(y, X[c])[0]) >= rt]
    final = list(rel)
    for i in range(len(rel)):
        for j in range(i+1, len(rel)):
            if rel[i] in final and rel[j] in final:
                if abs(X[rel[i]].corr(X[rel[j]])) >= ct:
                    if abs(pointbiserialr(y,X[rel[i]])[0]) < abs(pointbiserialr(y,X[rel[j]])[0]):
                        final.remove(rel[i])
                    else: final.remove(rel[j])
    if len(final) < 5: final = sorted(rel, key=lambda c: abs(pointbiserialr(y,X[c])[0]), reverse=True)[:5]
    return final

def shap_refine(X, y, cands, thr, mn=5):
    if len(cands) <= mn: return cands
    clf = LGBMClassifier(**LGBM_SFS).fit(X[cands], y)
    sv = shap.TreeExplainer(clf).shap_values(X[cands])
    if isinstance(sv,list): sv=sv[1]
    if sv.ndim==3: sv=sv[:,:,1]
    imp = pd.Series(np.abs(sv).mean(axis=0), index=cands)
    t = imp.sum()
    if t==0: return cands
    keep = (imp/t)[(imp/t)>=thr].index.tolist()
    return keep if len(keep)>=mn else imp.head(mn).index.tolist()

def stability_check(X, y, pl, cands, thr, mn=5):
    if len(cands)<=mn: return cands, 1.0
    projs = list(pl.unique())
    tk = max(1, math.ceil(0.7*len(cands)))
    pts = {}
    for p in projs:
        m = pl==p
        Xp, yp = X.loc[m,cands], y.loc[m]
        if len(Xp)<20 or yp.value_counts().min()<2: continue
        c = LGBMClassifier(**LGBM_SFS).fit(Xp, yp)
        s = shap.TreeExplainer(c).shap_values(Xp)
        if isinstance(s,list): s=s[1]
        if s.ndim==3: s=s[:,:,1]
        r = pd.Series(np.abs(s).mean(axis=0), index=cands).sort_values(ascending=False)
        pts[p] = set(r.head(tk).index)
    if not pts: return cands, 1.0
    ne = len(pts)
    pres = {f: sum(f in s for s in pts.values())/ne for f in cands}
    sel = [f for f in cands if pres[f]>=thr]
    if len(sel)<mn: sel = sorted(cands, key=lambda f: pres[f], reverse=True)[:mn]
    pn = list(pts.keys())
    if len(pn)>=2:
        jacs = [len(pts[a]&pts[b])/len(pts[a]|pts[b]) if pts[a]|pts[b] else 1 for a,b in combinations(pn,2)]
        mj = float(np.mean(jacs))
    else: mj=1.0
    return sel, mj

def borutashap(X, y, max_iter=10, alpha=0.05, mn=5):
    if X.shape[1]<=mn: return list(X.columns)
    sn = [f'sh_{c}' for c in X.columns]
    Xs = X.copy()
    for c in X.columns: Xs[f'sh_{c}'] = np.random.permutation(X[c].values)
    conf, tent = set(), set(X.columns)
    for _ in range(max_iter):
        if len(tent)<=mn: break
        feats = list(conf|tent)+sn
        clf = LGBMClassifier(**LGBM_SFS).fit(Xs[feats], y)
        sv = shap.TreeExplainer(clf).shap_values(Xs[feats])
        if isinstance(sv,list): sv=sv[1]
        if sv.ndim==3: sv=sv[:,:,1]
        imp = pd.Series(np.abs(sv).mean(axis=0), index=feats)
        smax = imp[sn].max()
        hits = {f: imp.get(f,0)>smax for f in tent}
        nh = sum(hits.values()); nt = len(hits)
        if nt==0: break
        pv = 1-stats.binom.cdf(nh-1,nt,0.5)
        if pv<alpha:
            for f,h in hits.items():
                if h: conf.add(f)
            tent -= conf
        tent -= set(sn)
    res = list(conf|tent)
    if len(res)<mn:
        ai = pd.Series(np.abs(sv).mean(axis=0), index=list(X.columns))
        res = ai.nlargest(mn).index.tolist()
    return res

def lopo_folds(pool):
    projs = list(pool.keys())
    common = None
    for p in projs:
        cs = set(pool[p].columns)-{'defective'}
        common = cs if common is None else common&cs
    common = sorted(common)
    for tgt in projs:
        srcs = pd.concat([pool[p][common+['defective']] for p in projs if p!=tgt], ignore_index=True)
        test = pool[tgt][common+['defective']].copy()
        ytr = srcs.pop('defective'); yte = test.pop('defective')
        sc = MinMaxScaler()
        Xtr = pd.DataFrame(sc.fit_transform(srcs), columns=common)
        Xte = pd.DataFrame(sc.transform(test), columns=common)
        pl = pd.Series(np.concatenate([[p]*len(pool[p][common]) for p in projs if p!=tgt]))
        yield {'target':tgt,'X_train':Xtr,'y_train':ytr,'X_test':Xte,'y_test':yte,'project_labels':pl}

## 3. Run Extended Evaluation

In [ ]:
def run_all(pool, name):
    models = {
        'lightgbm': ('lightgbm', LGBMClassifier(**LGBM_FINAL)),
        'random_forest': ('random_forest', RandomForestClassifier(**RF_FINAL)),
        'svm': ('svm', SVC(**SVM_FINAL)),
    }
    methods = ['AEFS','Ablation_no2D','AllFeatures','FilterMI','WrapperRFE','BorutaSHAP']
    all_rows = []
    folds = list(lopo_folds(pool))
    print(f'[{name}] {len(folds)} folds x {len(methods)} methods x {len(models)} models')

    for i, fold in enumerate(folds, 1):
        t0 = time.time()
        tgt = fold['target']
        Xtr, ytr = fold['X_train'], fold['y_train']
        Xte, yte = fold['X_test'], fold['y_test']
        pl = fold['project_labels']

        tw = Xtr.copy(); tw['defective'] = ytr.values
        cands = rup_filter(tw)
        sf = sequential_forward_selection(Xtr, ytr, cands)
        sf = shap_refine(Xtr, ytr, sf, 0.02)
        aefs, jac = stability_check(Xtr, ytr, pl, sf, 0.30)
        k = max(1, min(len(aefs), Xtr.shape[1]))
        mi = SelectKBest(mutual_info_classif, k=k).fit(Xtr, ytr)
        mi_f = Xtr.columns[mi.get_support()].tolist()[:k]
        rfe = RFE(LGBMClassifier(**LGBM_SFS), n_features_to_select=k).fit(Xtr, ytr)
        rfe_f = Xtr.columns[rfe.support_].tolist()
        bor_f = borutashap(Xtr, ytr)

        mf = {'AEFS':aefs,'Ablation_no2D':sf,'AllFeatures':list(Xtr.columns),
              'FilterMI':mi_f,'WrapperRFE':rfe_f,'BorutaSHAP':bor_f}
        mj = {'AEFS':jac}

        for m in methods:
            feats = mf[m]
            j = mj.get(m)
            for ml,(lbl,tmpl) in models.items():
                clf = tmpl.__class__(**tmpl.get_params())
                clf.fit(Xtr[feats], ytr)
                proba = clf.predict_proba(Xte[feats])[:,1]
                pred = (proba>=0.5).astype(int)
                mcc = matthews_corrcoef(yte, pred)
                f1v = f1_score(yte, pred, zero_division=0)
                try: auc = roc_auc_score(yte, proba)
                except: auc = float('nan')
                all_rows.append({
                    'pool':name,'target':tgt,'method':m,'model':lbl,
                    'n_features':len(feats),
                    'mean_jaccard':round(j,4) if j is not None else np.nan,
                    'test_mcc':round(mcc,4),'test_f1':round(f1v,4),'test_auc_roc':round(auc,4),
                })
        dt = time.time()-t0
        print(f'  {i}/{len(folds)}: {tgt} done ({dt:.1f}s) AEFS={len(aefs)} BorutaSHAP={len(bor_f)}')

    df = pd.DataFrame(all_rows)
    df.to_csv(f'{RESULTS_DIR}/{name}_extended_results.csv', index=False)
    print(f'Saved {len(df)} rows to {RESULTS_DIR}/{name}_extended_results.csv')
    return df

In [ ]:
t0 = time.time()
r_aeeem = run_all(aeeem, 'aeeem')
r_nasa = run_all(nasa, 'nasa')
r_relink = run_all(relink, 'relink')
r_tera = run_all(tera, 'tera')
print(f'\nTotal time: {(time.time()-t0)/60:.1f} minutes')

## 4. Summary

In [ ]:
methods = ['AEFS','Ablation_no2D','AllFeatures','FilterMI','WrapperRFE','BorutaSHAP']
for ds, df in [('AEEEM',r_aeeem),('TeraPromise',r_tera),('NASA',r_nasa),('ReLink',r_relink)]:
    print(f'\n=== {ds} ===')
    for ml in ['lightgbm','random_forest','svm']:
        sub = df[df['model']==ml]
        print(f'  {ml}:')
        for m in methods:
            s = sub[sub['method']==m]
            if len(s)==0: continue
            j = s['mean_jaccard'].dropna().mean()
            js = f'{j:.3f}' if not np.isnan(j) else ' ---'
            print(f'    {m:20s} n={s["n_features"].mean():5.1f} J={js:6s} MCC={s["test_mcc"].mean():.3f}+/-{s["test_mcc"].std():.3f} F1={s["test_f1"].mean():.3f} AUC={s["test_auc_roc"].mean():.3f}')

## 5. Statistical Tests

In [ ]:
print('WILCOXON: AEFS vs baselines (LightGBM)')
print('='*70)
for ds, df in [('AEEEM',r_aeeem),('TeraPromise',r_tera),('NASA',r_nasa),('ReLink',r_relink)]:
    print(f'\n--- {ds} ---')
    aefs = df[(df['method']=='AEFS')&(df['model']=='lightgbm')]['test_mcc'].values
    for bl in methods[1:]:
        blv = df[(df['method']==bl)&(df['model']=='lightgbm')]['test_mcc'].values
        if len(aefs)==len(blv) and len(aefs)>2:
            try:
                stat,pval = stats.wilcoxon(aefs, blv, alternative='two-sided')
                sig = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else 'ns'
                n1 = len(aefs)
                delta = (sum(1 for a in aefs for b in blv if a>b)-sum(1 for a in aefs for b in blv if a<b))/(n1*len(blv))
                eff = 'negl.' if abs(delta)<0.147 else 'small' if abs(delta)<0.33 else 'medium' if abs(delta)<0.474 else 'large'
                print(f'  vs {bl:20s}: p={pval:.4f} {sig} delta={delta:+.3f} ({eff})')
            except: print(f'  vs {bl:20s}: error')
        else: print(f'  vs {bl:20s}: n={len(aefs)} (too few)')

## 6. Upload to HuggingFace

In [ ]:
from huggingface_hub import HfApi, login
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')
if not HF_TOKEN: raise RuntimeError('HF_TOKEN not found')
login(token=HF_TOKEN)
api = HfApi()
HF_REPO = 'MoshinAli/aefs-cpdp-results'
try: api.create_repo(HF_REPO, repo_type='dataset', private=True, exist_ok=True)
except: pass
print(f'Repo: https://huggingface.co/datasets/{HF_REPO}')

In [ ]:
upload_files = [f for f in os.listdir(RESULTS_DIR) if f.endswith('.csv') and 'extended' in f]
print(f'Uploading {len(upload_files)} files...')
for f in upload_files:
    fp = os.path.join(RESULTS_DIR, f)
    hf_path = f'expanded/extended/{f}'
    try:
        api.upload_file(path_or_fileobj=fp, path_in_repo=hf_path, repo_id=HF_REPO, repo_type='dataset')
        print(f'  OK: {hf_path}')
    except Exception as e:
        print(f'  FAIL: {hf_path} -- {e}')
print('Done!')